**Importing required libraries and mounting Google Drive to access the dataset.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os
import cv2
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import seaborn as sns

**Defining the dataset paths for original and augmented CT images of kidney stone and non-stone classes.**

In [ ]:
original_stone = '/content/drive/MyDrive/KINDYS~1/Original Dataset/Stone'
original_nonstone = '/content/drive/MyDrive/KINDYS~1/Original Dataset/Non-Stone'

aug_stone = '/content/drive/MyDrive/KINDYS~1/Augmented Dataset/Stone'
aug_nonstone = '/content/drive/MyDrive/KINDYS~1/Augmented Dataset/Non-Stone'

**Loading images from the dataset folders, converting them to grayscale, resizing them, and assigning labels.**

In [ ]:
images = []
labels = []

def load_folder(path, label):
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, (128,128))

        images.append(img)
        labels.append(label)

load_folder(original_stone, 1)
load_folder(original_nonstone, 0)

load_folder(aug_stone, 1)
load_folder(aug_nonstone, 0)

images = np.array(images)
labels = np.array(labels)

print(images.shape)
print(labels.shape)

**Splitting the dataset into training and testing sets**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

**Normalizing image pixel values and reshaping the dataset to fit the CNN input format.**

In [ ]:
x_train = x_train.reshape((-1,128,128,1)).astype("float32")/255
x_test = x_test.reshape((-1,128,128,1)).astype("float32")/255

y_train = to_categorical(y_train,2)
y_test = to_categorical(y_test,2)

print(x_train.shape)
print(y_train.shape)

**Building the convolutional neural network architecture for kidney stone classification.**

In [ ]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(128,128,1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64,(3,3),activation='relu',padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128,(3,3),activation='relu',padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(2,activation='softmax')
])

**Compiling the CNN model using Adam optimizer and categorical crossentropy loss.**

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Training the CNN model with early stopping to prevent overfitting.**

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

**Evaluating the trained model on the test dataset.**

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

**Visualizing training and validation accuracy and loss during model training.**

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train','Validation'])

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy over epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train','Validation'])

plt.show()

**Generating predictions on the test dataset and computing the confusion matrix.**

In [ ]:
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred,axis=1)

y_true = np.argmax(y_test,axis=1)

cm = confusion_matrix(y_true,y_pred_classes)

**Displaying the confusion matrix and classification report for model performance evaluation.**

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Non-Stone','Stone'],
            yticklabels=['Non-Stone','Stone'])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

print(classification_report(y_true,y_pred_classes))

**Saving the trained CNN model for future inference and deployment.**

In [ ]:
model.save('/content/drive/MyDrive/KINDYS~1/kidney_stone_cnn_final.h5')